# McCormick緩和と空間分枝木 — 診断エンジンはどこを分枝しているかを見る

`weak_relaxation`(緩和が弱い)診断が根拠にする2つの観測量のうち、片方
(`spatial_share`=双対境界改善に占める空間分枝の寄与)は **空間分枝木の収集器**
(`minlpkit.collectors.tree`)が作る。この notebook はその収集器が何を記録し、
どう可視化されるかを手を動かして追う。

1. **McCormick緩和とは何か** — 双線形項 $z=xy$ の凸包絡が区間分割でどう締まるか(原理)
2. **何を見るか** — `NODEBRANCHED` イベントが記録する(ノード番号/親/深さ/下界/分枝変数と型)
3. **可視化** — 実際に解いて分枝木を tidy tree レイアウトで描く。連続変数への分枝(spatial)を色分け
4. **読み解き** — spatial分枝の集中先と双対境界の寄与を突き合わせる

題材は **バッチ反応器スケジューリング**(`samples/others/scheduling_plant.py`)。
バッチサイズ `s`・反応温度 `T`・反応時間 `tau` などの連続変数どうしの積を多数含み、
300秒でgap72.5%まで停滞する診断済みの難問(`.claude/skills/minlp-run/SKILL.md`)。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.tree import solve_and_collect
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind
from viz.mccormick import Box, max_gap, piecewise_underestimator, true_surface

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
KIND_COLOR = {"spatial": C["s1"], "integer": C["s2"], "binary": "#e87ba4", "root": C["ink"]}
KIND_LABEL = {"spatial": "空間分枝(連続)", "integer": "整数分枝", "binary": "0-1分枝", "root": "根"}
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. McCormick緩和の原理 — 区間分割でなぜ締まるか

双線形項 $z = xy$($x,y$ ともに連続)を SCIP は一般に **McCormick凸包絡**で下から抑える。
box $[x_L,x_U]\times[y_L,y_U]$ 全体を1つの緩和で覆うと、鞍型の真の曲面とのギャップが大きい。
box を分割して**それぞれの小さい box に McCormick を張る**(区分McCormick)と、
分割数 $k$ を増やすほどギャップは $O(1/k)$ で縮む。**この「区間を割って緩和を締める」操作こそが
SCIPの空間分枝そのもの**であり、次節で見る `NODEBRANCHED` イベントの正体である。

In [2]:
BOX: Box = (-2.0, 2.0, -2.0, 2.0)
KS = [1, 2, 4, 8]
xg = np.linspace(BOX[0], BOX[1], 60)
yg = np.linspace(BOX[2], BOX[3], 60)
Zt = true_surface(xg, yg)

frames = [go.Frame(
    data=[go.Surface(x=xg, y=yg, z=Zt, showscale=False, opacity=0.5,
                     colorscale=[[0, C["muted"]], [1, C["muted"]]],
                     name="真の曲面 z=x*y", hoverinfo="skip"),
          go.Surface(x=xg, y=yg, z=piecewise_underestimator(xg, yg, BOX, k),
                     showscale=False, opacity=0.95,
                     colorscale=[[0, "#9ec5f4"], [1, C["s1"]]],
                     name="区分McCormick下界",
                     hovertemplate="下界 %{z:.2f}<extra></extra>")],
    name=str(k), layout=go.Layout(title=dict(
        text=f"box を <b>{k}分割</b>して張った McCormick 下界 — 最大ギャップ {max_gap(BOX, k):.3f}")))
    for k in KS]

fig = go.Figure(data=frames[0].data, frames=frames)
steps = [dict(method="animate", label=f"{k}分割",
             args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
        for k in KS]
fig.update_layout(
    title=dict(text=frames[0].layout.title.text, font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], font=dict(family=FONT, color=C["ink2"], size=12),
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
              bgcolor=C["surface"]),
    height=460, margin=dict(l=10, r=10, t=48, b=10),
    sliders=[dict(active=0, x=0.1, len=0.85, y=0,
                  currentvalue=dict(prefix="分割数: ", font=dict(color=C["ink2"])),
                  steps=steps)])
show(fig)

gaps = [max_gap(BOX, k) for k in [1, 2, 3, 4, 6, 8, 12, 16]]
print("k -> 最大ギャップ:", dict(zip([1, 2, 3, 4, 6, 8, 12, 16], [f"{g:.3f}" for g in gaps])))

k -> 最大ギャップ: {1: '3.982', 2: '1.991', 3: '1.327', 4: '0.995', 6: '0.664', 8: '0.498', 12: '0.332', 16: '0.249'}


分割数 $k$ を増やすほど最大ギャップは単調に縮む($\Delta x\,\Delta y/(4k)$ に比例)。
**SCIPの空間分枝は、この「区間を割る」操作を探索木の中で連続変数ごとに動的に行っている。**
どの変数の、どの深さで、どれだけ空間分枝が起きたかを記録するのが次節の収集器。

## 2. 何を見るか — `NODEBRANCHED` イベントの収集

`minlpkit.collectors.tree.TreeCollector` は SCIP の `NODEBRANCHED` イベントを毎回捕まえ、

- `node` / `parent` / `depth` — 木の位置
- `lowerbound` — そのノードの局所下界(=そのノードでの双対境界)
- `branch_var` / `kind` — 分枝に使われた変数と型。`kind` は変数の `vtype()` から
  `spatial`(連続=空間分枝)/ `integer` / `binary` に分類する

を1行ずつ `DataFrame` にする。`layout_tree` はこれを tidy tree レイアウト(葉をDFS順に並べ、
内部ノードは子の平均位置)に変換する。**技術的な注意**: 分枝は変換後変数に対して起きるため、
型は分枝が返す変数オブジェクトの `.vtype()` から取る(元変数名の辞書引きは名前不一致で失敗する)。

In [3]:
df = solve_and_collect(sp.build_model(), max_nodes=400, node_limit=3000)
print(f"収集した分枝ノード数: {len(df)} / 最大深さ: {int(-df['y'].min())}")
df[["node", "parent", "depth", "lowerbound", "branch_var", "kind"]].head(8)

収集した分枝ノード数: 400 / 最大深さ: 15


,node,parent,depth,lowerbound,branch_var,kind
0,1,NaN,0,52.128421,NaN,root
1,3,1.0,1,54.462172,t_x_J4_M2,binary
2,4,3.0,2,69.680195,t_n_J4,integer
3,7,4.0,3,71.102373,t_x_J6_M2,binary
4,9,7.0,4,80.828859,t_x_J5_M2,binary
5,10,9.0,5,99.390241,t_n_J2,integer
6,2,1.0,1,52.554462,t_x_J4_M2,binary
7,14,2.0,2,71.855301,t_n_J2,integer


## 3. 可視化 — 分枝木を tidy tree で描く

各点が1回の分枝ノード。下方向が探索の深さ。色は分枝変数の型。
**青(spatial)が下の方(深い階層)に集中しているか**が、このモデルの難しさの見どころ。

In [4]:
def fig_tree(d: pd.DataFrame) -> go.Figure:
    xmap, ymap = dict(zip(d["node"], d["x"])), dict(zip(d["node"], d["y"]))
    ex, ey = [], []
    for _, r in d.iterrows():
        if r["parent"] in xmap:
            ex += [xmap[r["parent"]], r["x"], None]
            ey += [ymap[r["parent"]], r["y"], None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color=C["grid"], width=1),
                             hoverinfo="skip", showlegend=False))
    for kind in ["spatial", "integer", "binary", "root"]:
        dd = d[d["kind"] == kind]
        if dd.empty:
            continue
        fig.add_trace(go.Scatter(
            x=dd["x"], y=dd["y"], mode="markers", name=KIND_LABEL[kind],
            marker=dict(color=KIND_COLOR[kind], size=8 if kind == "root" else 6,
                       line=dict(color=C["surface"], width=1),
                       symbol="square" if kind == "root" else "circle"),
            customdata=dd[["node", "depth", "lowerbound", "branch_var"]],
            hovertemplate="ノード%{customdata[0]} / 深さ%{customdata[1]}<br>"
                         "下界 %{customdata[2]:.2f}<br>分枝: %{customdata[3]}<extra></extra>"))
    inc = d[d["incumbent"]]
    if not inc.empty:
        fig.add_trace(go.Scatter(x=inc["x"], y=inc["y"], mode="markers", name="暫定解発見",
            marker=dict(color="#eda100", size=12, symbol="star", line=dict(color=C["surface"], width=1))))
    fig.update_layout(
        title=dict(text="空間分枝木 — 分枝変数の型で色分け(下方向=深さ)",
                  font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(visible=False),
        yaxis=dict(title="深さ", gridcolor=C["grid"], linecolor=C["axis"],
                  tickfont=dict(color=C["muted"]), zeroline=False),
        margin=dict(l=50, r=16, t=48, b=16), height=480, hovermode="closest",
        legend=dict(orientation="h", y=1.0, yanchor="bottom", x=1.0, xanchor="right",
                   font=dict(size=11, color=C["ink2"])))
    return fig

show(fig_tree(df))

In [5]:
counts = df[df["kind"] != "root"]["kind"].value_counts()
order = ["spatial", "integer", "binary"]
vals = [int(counts.get(k, 0)) for k in order]
fig = go.Figure(go.Bar(x=vals, y=[KIND_LABEL[k] for k in order], orientation="h",
    marker=dict(color=[KIND_COLOR[k] for k in order]), text=vals, textposition="outside"))
fig.update_layout(
    title=dict(text="分枝回数の内訳", font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="分枝回数", gridcolor=C["grid"], linecolor=C["axis"], tickfont=dict(color=C["muted"])),
    margin=dict(l=110, r=40, t=44, b=40), height=220)
show(fig)
print(f"spatial={vals[0]} integer={vals[1]} binary={vals[2]} (計{sum(vals)}回)")

spatial=75 integer=149 binary=175 (計399回)


## 4. 読み解き — spatial分枝は双対境界の何%を押し上げたか

木の「分枝回数」だけでは「効いたかどうか」は分からない。`minlpkit.collectors.attribution` は
同じ `kind` 分類を使い、各分枝の直後に生じた **双対境界の増分 `Δdual`** をその分枝に帰属させる。
`spatial_share`(空間分枝の寄与)は `weak_relaxation` 診断ルールが直接見る観測量そのもの。

In [6]:
d, summ = solve_and_attribute(sp.build_model(), time_limit=15, gap_limit=0.01)
gk = gain_by_kind(d)
total = gk["dual_gain"].sum()
spatial = gk.loc[gk["kind"] == "spatial", "dual_gain"].sum()
print(gk.to_string(index=False))
print(f"\nspatial_share = {spatial/total:.1%}  (これが report.metrics['spatial_share'] の中身)")

fired = mk.evaluate(dict(bottleneck_rel_viol=1.0, spatial_share=spatial/total))
print("weak_relaxation 発火:", any(f["id"] == "weak_relaxation" for f in fired))

   kind  dual_gain
spatial  23.709919
integer  18.724105
 binary   1.476824
   root   0.000000

spatial_share = 54.0%  (これが report.metrics['spatial_share'] の中身)
weak_relaxation 発火: True


## まとめ

- 空間分枝木の収集器は `NODEBRANCHED` イベントを型別(spatial/integer/binary)に集めるだけの
  単純な仕掛けだが、これが `spatial_share` という診断ルールの核心的な観測量を作る。
- 分枝**回数**の内訳(棒グラフ)と、双対境界**改善への寄与**(attribution)は別物 — 回数が多くても
  寄与が小さいことはあり得るため、診断は寄与の方(`gain_by_kind`)を見る。
- `spatial_share` が高い(=非線形項の凸緩和を締める作業が支配的)ときに `weak_relaxation` が
  発火し、`mk.linearize_product` / `mk.pwl_sos2` による再定式化が推薦される
  (深掘りは[整数×連続の厳密線形化](../improve/01_linearize_product.ipynb))。

関連: [手法ガイド 0. 診断そのもの](../../playbook/00-diagnose.md) /
API [`minlpkit.collectors.tree`](../../api/pipeline.md)。